# 14 · Событийная панель v3 из сырого журнала (CPU)

**Вычисления:** CPU, **Accelerator = None**. Интернет не нужен, если в образе есть `7z` (обычно есть);
иначе ноутбук поставит `py7zr` — тогда включите Internet в Settings.

Ноутбук заменяет дневную панель v2 (одна смешанная цель «Неисправен/Обесточен в D+2») на панель, где каждая
цель ТЗ считается отдельно по **секундному** журналу:

| цель | что значит | тип оценки |
|---|---|---|
| `target_t2a_link_onset` | начало «Неисправен» (= потеря связи, FAQ организаторов) в окне D+2 | onset proxy |
| `target_t2b_link_sustained` | то же, но эпизод длится ≥ τ (5 мин) | conditional proxy, цензура → null |
| `target_t2c_power_onset` / `_sustained` | начало «Обесточен» / длительностью ≥ τ | onset proxy |
| `target_t1a_tech_value` | технический код или значение вне диапазона у числового канала | data-quality proxy |
| `target_t4_gas_cross` | газ пересекает 1 % снизу вверх (строго) | threshold event |

Главные правила (подробно — `docs/EVENT_PANEL_V3_RU.md`):
- одна секунда канала — неупорядоченное множество значений; конфликт внутри секунды = неоднозначно, не onset;
- признаки на конец D — только прошлое; eligibility — состояние на конец D; старт в D+1 — competing outcome;
- метка решается не позже D+4 (L = 24 ч) → `d_label_decision_end` для purging в ноутбуке 15;
- тишина в окне — не отрицательный пример (`unobserved_window` → null); неоднозначное → null с причиной;
- поток = (канал, класс сигнала), stream mapping v1: «O» связи — только сообщения функции самого канала, «O» питания —
  только явные «Есть питание»/«Питание от сети»; «Неопределен» — неизвестное состояние связи (U).
  Две заранее объявленные sensitivity-версии считаются отдельно, в выход идут только числа изменившихся меток.

**Выход** (`/kaggle/working`): `event_panel_v3/` (части parquet + `panel_manifest_v3.json` с SHA-256) — вход ноутбука 15;
`audit_event_panel_v3.json` и `summary_event_panel_v3_ru.md` — только агрегаты. Ключи каналов и объектов
псевдонимизированы (`sha256(соль + id)[:16]`), сырых идентификаторов и значений в выходе нет.


## Как запустить

1. **Add Input** → **приватный** dataset с журналами `ext-journal-2019 … ext-journal-2026` (`.7z`, `.csv` или папки,
   в которые Kaggle их распаковал) и справочником каналов. Путь не нужен — файлы ищутся во всех входах.
   Kaggle заменяет кириллицу в имени файла на `_` (справочник превращается в `__.csv`) — справочник и журналы
   дополнительно распознаются по заголовку CSV. Ячейка «Проверка входов» остановит прогон, если чего-то нет.
   Старая обезличенная панель v2 **не подходит**: в ней нет истории событий. Локально можно задать `LDT_KAGGLE_INPUT_DIR`.
2. **Settings → Accelerator → None.**
3. Сначала `MODE = "SMOKE"` (2019–2020, ≈1/20 каналов, пара минут) — цифры `non_comparable`.
4. Затем `MODE = "FULL"` и **Save Version → Save & Run All**, чтобы выход сохранился и его можно было
   подключить к ноутбуку 15 как Input.


In [ ]:
# ЕДИНСТВЕННАЯ ЯЧЕЙКА, КОТОРУЮ МОЖНО ИЗМЕНЯТЬ
MODE = "FULL"        # "FULL" или "SMOKE"
N_BUCKETS = 4        # групп каналов: больше — меньше памяти, дольше
TAU_MINUTES = 5      # порог устойчивого эпизода для *_sustained
RUN_SENSITIVITY = True  # + 2 пересборки разметки (≈×3 времени): «Неопределен» нейтрален; сообщения функции = O питания
print("Режим:", MODE, "| групп каналов:", N_BUCKETS, "| τ =", TAU_MINUTES, "мин | sensitivity:", RUN_SENSITIVITY)


In [ ]:
import os, shutil, sys, platform, warnings
if os.name == "nt":  # локальный запуск на Windows: вывод кириллицы
    for stream in (sys.stdout, sys.stderr):
        try:
            stream.reconfigure(encoding="utf-8", errors="replace")
        except AttributeError:
            pass
import polars as pl
# шумные, но безвредные предупреждения polars (join_asof по группам; смена умолчания explode в 2.0)
warnings.filterwarnings("ignore", message=".*Sortedness of columns.*")
warnings.filterwarnings("ignore", message=".*empty_as_null.*")
print("python  ", platform.python_version())
print("polars  ", pl.__version__)
print("CPU     ", os.cpu_count())
try:
    mem = os.sysconf("SC_PAGE_SIZE") * os.sysconf("SC_PHYS_PAGES") / 2**30
    print(f"RAM      {mem:.1f} ГБ")
except (AttributeError, ValueError, OSError):
    pass
for d in ("/kaggle/working", "/tmp"):
    if os.path.exists(d):
        print(f"диск {d}: свободно {shutil.disk_usage(d).free / 2**30:.1f} ГБ")
print("7z      ", next((shutil.which(x) for x in ("7z", "7za", "7zr") if shutil.which(x)), "нет (будет py7zr)"))


## Таксономия значений (встроена, SHA-256 попадает в manifest)


In [ ]:
import json, os
from pathlib import Path
TAXONOMY_JSON = '{\n  "version": "3.1",\n  "description": "Классы значений значение_датчика для событийной панели v3. Не метки физических отказов. Источник: справочник_состояний.csv (организаторы) + словарь значений журнала; расхождения со справочником отмечены в notes.",\n  "text_classes": {\n    "link_fault": [\n      "Неисправен"\n    ],\n    "power_off": [\n      "Обесточен"\n    ],\n    "power_on": [\n      "Есть питание",\n      "Питание от сети"\n    ],\n    "power_degraded": [\n      "Питание от батарей",\n      "Батарея разряжена",\n      "Батарея неисправна"\n    ],\n    "device_off": [\n      "Отключено устройство"\n    ],\n    "service": [\n      "Неопределен"\n    ],\n    "undefined_measurement": [\n      "Не определено"\n    ],\n    "object_summary": [\n      "Много неисправных устройств",\n      "Устройства на объекте исправны",\n      "Работают все насосы АНС",\n      "Работают все насосы в АНС",\n      "Включены все насосы АНС"\n    ],\n    "security_mode": [\n      "На охране",\n      "Снято с охраны"\n    ],\n    "flood": [\n      "Затоплен"\n    ],\n    "event_alarm": [\n      "Обнаружено движение",\n      "Движение вверх",\n      "Движение вниз",\n      "Движение влево",\n      "Движение вправо",\n      "Обнаружен дым",\n      "Обнаружен газ",\n      "Не замкнут",\n      "Замкнут",\n      "Рычаг сдернут",\n      "Рычаг сдернут влево",\n      "Рычаг сдернут вправо",\n      "Оба рычага сдернуты",\n      "Вызов",\n      "Разговор",\n      "Температура выше 35ºC1",\n      "Температура выше 40ºC",\n      "Температура выше 27ºC",\n      "Температура ниже 3ºC",\n      "Температура ниже 3ºC1"\n    ],\n    "normal": [\n      "Норма",\n      "Движения нет",\n      "Дыма нет",\n      "Рычаг норма",\n      "Включен",\n      "Выключен",\n      "В норме от +3 до +40",\n      "В норме от +3 до +351",\n      "В норме от +3 до +27ºC"\n    ]\n  },\n  "numeric": {\n    "technical_value_candidates": [\n      -127.0,\n      -100.0,\n      255.0,\n      -3276.0,\n      -3276.8,\n      327.68,\n      3276.7\n    ],\n    "gas_sensor_type": "Газовый датчик",\n    "gas_out_of_range": {\n      "below": 0.0,\n      "above": 100.0\n    },\n    "gas_threshold_percent": 1.0,\n    "gas_warning_percent": 5.0,\n    "gas_band_low": 0.1\n  },\n  "date_like": {\n    "regex": "^\\\\d{2}\\\\.\\\\d{2}\\\\.\\\\d{4}",\n    "clock_1970_prefix": "01.01.1970",\n    "duplicate_timestamp_tolerance_seconds": 5\n  },\n  "notes": [\n    "Неисправен = «только факт потери связи с устройством» (FAQ организаторов); в справочнике есть только у набора 1 (КД Дверь), в журнале — у насосов, вентиляторов, дыма, газа, фаз.",\n    "Обесточен у «Состояния фазы» в справочнике отсутствует; у набора 10 (Датчик дыма) помечен тревожное=false.",\n    "−100, 255 — примеры технических кодов производителей из FAQ, не универсальные пороги; кандидаты проверяются агрегатами на train.",\n    "01.01.1970 — ошибка часов/формата (FAQ: «скорее всего … сбилась дата»), не метка отказа измерителя; даты, совпадающие со временем события, — дубль timestamp.",\n    "Связь: O — только сообщения функции самого канала (норма/событие/число); питание, охрана, затопление, служебные — не O связи.",\n    "Питание: O — только явные «Есть питание»/«Питание от сети»; «Питание от батарей» нейтрально; во время «Неисправен» питание неизвестно (U).",\n    "«Неопределен» в основной версии — U для связи (не O и не восстановление); sensitivity-версия считает его нейтральным (GPT 16/Q1).",\n    "Коды T1a (−127, −100, 255, −3276, −3276.8, 327.68, 3276.7) — фиксированное правило из FAQ и данных 2019 до любой валидации; не означают физический отказ датчика."\n  ],\n  "stream_mapping": {\n    "version": "1.0",\n    "note": "stream_id = (канал, класс сигнала). F — отказ потока, O — однозначное рабочее состояние того же потока. F+O в одной секунде → A (conflict). U-значение: при unknown_overrides=true делает секунду A всегда, иначе — только если в секунде нет F/O этого потока. Значения других классов поток не меняют (GPT 16: O другого класса не закрывает F связи; «Обесточен» не O связи).",\n    "streams": {\n      "link": {\n        "fault": [\n          "link_fault"\n        ],\n        "ok": [\n          "normal",\n          "event_alarm",\n          "numeric"\n        ],\n        "unknown": [\n          "service"\n        ],\n        "unknown_overrides": true\n      },\n      "power": {\n        "fault": [\n          "power_off"\n        ],\n        "ok": [\n          "power_on"\n        ],\n        "unknown": [\n          "link_fault"\n        ],\n        "unknown_overrides": false\n      }\n    },\n    "sensitivity": {\n      "service_neutral": "link.unknown без service: «Неопределен» не меняет состояние связи"\n    }\n  }\n}\n'
TAXONOMY_PATH = Path(os.environ.get("LDT_STAGE_DIR", "/tmp/ldt_event_panel_v3_stage")).parent / "state_taxonomy_v3.json"
TAXONOMY_PATH.parent.mkdir(parents=True, exist_ok=True)
TAXONOMY_PATH.write_text(TAXONOMY_JSON, encoding="utf-8")
print("таксономия:", json.loads(TAXONOMY_JSON)["version"], "| классов:", len(json.loads(TAXONOMY_JSON)["text_classes"]))


## Встроенный runtime панели v3

Код встроен; репозиторий на Kaggle не нужен.


In [ ]:
"""Событийная панель v3: state machine по сырому журналу → панель (канал × сутки D) с раздельными целями.

Спецификация согласована в переписке Claude ↔ GPT (Drive, документы 05 → 12) и docs/EVENT_PANEL_V3_RU.md.
Главные правила:
- одна секунда канала = неупорядоченное множество значений; порядок строк CSV и ид_события порядком не считаются;
- признаки на конец D — только прошлое; метка окна [D+2; D+3) может смотреть вперёд не дальше D+4 (L = 24 ч);
- eligibility на конец D задаётся только состоянием на конец D; старт в D+1 — competing outcome, а не фильтр;
- неоднозначное → null с причиной, а не 0;
- цели разные и не смешиваются: T2a (onset «Неисправен» = потеря связи), T2b (потеря связи ≥ τ),
  T2c (onset «Обесточен»), T2c_sustained, T1a (технические/диапазонные значения), T4 (газ ≥ 1 %).
Ничто здесь не является подтверждённым физическим отказом.
"""

from __future__ import annotations

import hashlib
import json
import re
from datetime import date, datetime, timedelta
from pathlib import Path
from typing import Any, Iterable

SCHEMA_VERSION = "3.0"
EVENT_COLUMNS = ["ид_события", "ид_канала_данных", "дата", "время", "тревожное", "значение_датчика"]
CATALOGUE_COLUMNS = ["ид_канала_данных", "тип_инж_системы", "тип_датчика", "ид_объект"]
KEY_SALT = "ldt-misis-panel-v3:"

DEFAULT_CONFIG: dict[str, Any] = {
    "tau_minutes": 5,                 # порог «устойчивого» эпизода; чувствительность 1/5/30/60 — отдельными прогонами
    "confirm_hours": 24,              # L: самое позднее подтверждение метки, ч
    "active_lookback_days": 7,        # строка (канал, D) есть, если канал слал события в [D-6; D]
    "tech_clean_days": 7,             # T1a: onset только после ≥ 7 суток без технических значений
    "numeric_history_days": 30,       # T1a: канал должен слать числа в последние 30 суток
    "window_offsets_days": [7, 30],   # окна скользящих признаков
    "service_as_unknown": True,       # «Неопределен» = U для связи (основная версия); False — sensitivity (GPT 16/Q1)
    "power_ok_from_function": False,  # sensitivity: сообщения функции канала (норма/событие/число) считаются O питания
}

# заранее объявленные sensitivity-варианты разметки (выбор между ними не делается по validation-метрикам)
SENSITIVITY_VARIANTS = {"service_neutral": {"service_as_unknown": False},
                        "power_ok_from_function": {"power_ok_from_function": True}}

LINK_STATES = ("O", "F", "A")  # ok / fault / ambiguous
TARGETS = ["target_t2a_link_onset", "target_t2b_link_sustained", "target_t2c_power_onset",
           "target_t2c_power_sustained", "target_t1a_tech_value", "target_t4_gas_cross"]
BASE_FEATURES = ["n_events", "n_alarm", "n_link_fault", "n_power_off", "n_service", "n_tech_like", "n_numeric",
                 "n_multi_value_seconds", "num_mean", "num_min", "num_max", "num_std", "num_last", "gas_max", "gas_last",
                 "link_state", "power_state", "d_weekday", "d_month"]


# ---------------------------------------------------------------------------
# 0. Таксономия и ключи
# ---------------------------------------------------------------------------
def load_taxonomy(path: Path) -> dict[str, Any]:
    tax = json.loads(Path(path).read_text(encoding="utf-8"))
    seen: dict[str, str] = {}
    for cls, values in tax["text_classes"].items():
        for v in values:
            if v in seen and seen[v] != cls:
                raise ValueError(f"значение {v!r} в двух классах: {seen[v]} и {cls}")
            seen[v] = cls
    tax["_value_to_class"] = seen
    tax["_sha256"] = hashlib.sha256(Path(path).read_bytes()).hexdigest()
    return tax


def pseudo_key(raw: str | None) -> str | None:
    if raw is None:
        return None
    return hashlib.sha256((KEY_SALT + str(raw)).encode("utf-8")).hexdigest()[:16]


def key_map(pl: Any, ids: Iterable[str], name: str) -> Any:
    ids = sorted({str(i) for i in ids if i is not None})
    return pl.DataFrame({"raw": ids, name: [pseudo_key(i) for i in ids]})


# ---------------------------------------------------------------------------
# 1. Классификация событий
# ---------------------------------------------------------------------------
def classify_events(pl: Any, frame: Any, tax: dict[str, Any], catalogue: Any) -> Any:
    """frame: сырые колонки журнала (строки). Возвращает ch, t, v, alarm, num, cls, is_gas, тип_*."""
    num_cfg, date_cfg = tax["numeric"], tax["date_like"]
    value = pl.col("значение_датчика").fill_null("")
    num = value.str.replace(",", ".", literal=True).cast(pl.Float64, strict=False)
    t = pl.concat_str(["дата", "время"], separator=" ").str.strptime(pl.Datetime("us"), "%Y-%m-%d %H:%M:%S", strict=False)
    out = (
        frame.join(catalogue, on="ид_канала_данных", how="left")
        .with_columns(
            pl.col("ид_канала_данных").alias("ch"), t.alias("t"), value.alias("v"), num.alias("num"),
            pl.col("тревожное").fill_null("").str.to_lowercase().is_in(["t", "true"]).alias("alarm"),
            (pl.col("тип_датчика") == num_cfg["gas_sensor_type"]).fill_null(False).alias("is_gas"),
        )
        .filter(pl.col("t").is_not_null())
    )
    is_date = pl.col("v").str.contains(date_cfg["regex"])
    parsed_date = pl.col("v").str.slice(0, 19).str.strptime(pl.Datetime("us"), "%d.%m.%Y %H:%M:%S", strict=False)
    tol = int(date_cfg["duplicate_timestamp_tolerance_seconds"])
    text_cls = pl.col("v").replace_strict(tax["_value_to_class"], default="unknown_value", return_dtype=pl.String)
    cls = (
        pl.when(pl.col("num").is_not_null() & pl.col("num").is_in(num_cfg["technical_value_candidates"])).then(pl.lit("tech_value"))
        .when(pl.col("num").is_not_null() & pl.col("is_gas")
              & ((pl.col("num") < num_cfg["gas_out_of_range"]["below"]) | (pl.col("num") > num_cfg["gas_out_of_range"]["above"])))
        .then(pl.lit("gas_out_of_range"))
        .when(pl.col("num").is_not_null()).then(pl.lit("numeric"))
        .when(is_date & pl.col("v").str.starts_with(date_cfg["clock_1970_prefix"])).then(pl.lit("clock_1970"))
        .when(is_date & ((parsed_date - pl.col("t")).dt.total_seconds().abs() <= tol)).then(pl.lit("dup_timestamp"))
        .when(is_date).then(pl.lit("date_other"))
        .when(pl.col("v") == "").then(pl.lit("empty_value"))
        .otherwise(text_cls)
    )
    return out.with_columns(cls.alias("cls")).select(
        "ch", "t", "v", "alarm", "num", "cls", "is_gas", "тип_датчика", "тип_инж_системы", "ид_объект")


# ---------------------------------------------------------------------------
# 2. Секундный уровень и точки смены состояния (RLE)
# ---------------------------------------------------------------------------
def second_table(pl: Any, ev: Any, tax: dict[str, Any]) -> Any:
    """Одна строка = (канал, секунда). Порядок внутри секунды не используется.
    Для каждого потока (связь, питание) секунда: F — только отказ, O — только рабочее состояние того же потока,
    A — отказ и O вместе или любое неизвестное (U) этого потока; null — секунда не несёт сведений о потоке."""
    streams = tax["stream_mapping"]["streams"]
    aggs = [pl.col("v").n_unique().alias("n_values"), (pl.col("cls") == "link_fault").any().alias("has_fault")]
    for name, st in streams.items():
        aggs += [pl.col("cls").is_in(st["fault"]).any().alias(f"_{name}_f"),
                 pl.col("cls").is_in(st["ok"]).any().alias(f"_{name}_o"),
                 pl.col("cls").is_in(st["unknown"]).any().alias(f"_{name}_u")]
    sec = ev.group_by("ch", "t").agg(*aggs)
    cols = []
    for name in streams:
        f, o, u = pl.col(f"_{name}_f"), pl.col(f"_{name}_o"), pl.col(f"_{name}_u")
        override = bool(streams[name].get("unknown_overrides", True))
        u_wins = u if override else (u & ~f & ~o)
        cols.append(pl.when(f & o).then(pl.lit("A")).when(u_wins).then(pl.lit("A"))
                    .when(f).then(pl.lit("F")).when(o).then(pl.lit("O")).otherwise(None).alias(f"{name}_cls"))
        cols.append(pl.when(f & o).then(pl.lit("conflict")).when(u_wins).then(pl.lit("unknown")).otherwise(None)
                    .alias(f"{name}_a_cause"))
    return sec.with_columns(cols).drop([c for c in sec.columns if c.startswith("_")])


def change_points(pl: Any, sec: Any, cls_col: str) -> Any:
    """RLE решающих секунд: строки, где класс сменился (по каналу, по времени)."""
    d = sec.filter(pl.col(cls_col).is_not_null()).select("ch", "t", pl.col(cls_col).alias("cls")).sort("ch", "t")
    return rle(pl, d)


def rle(pl: Any, d: Any) -> Any:
    d = d.sort("ch", "t")
    prev = pl.col("cls").shift(1).over("ch")
    return d.filter(prev.is_null() | (pl.col("cls") != prev)).select("ch", "t", "cls")


# ---------------------------------------------------------------------------
# 3. Эпизоды
# ---------------------------------------------------------------------------
def episodes(pl: Any, cp: Any, data_end: datetime, tau_minutes: float) -> Any:
    """Эпизод = RLE-блок класса F. onset_type: strict (перед ним однозначное O) или ambiguous.
    Длительность: до следующей точки смены. Следующая O → exact; A или конец данных → цензура (нижняя граница).
    sustained = 1 только при однозначном закрытии (O того же потока) после t0 + τ: повторы «Неисправен» и тишина
    непрерывность не доказывают (GPT 14, C7); незакрытый эпизод → null."""
    cp = cp.sort("ch", "t").with_columns(
        pl.col("cls").shift(1).over("ch").alias("prev_cls"),
        pl.col("cls").shift(-1).over("ch").alias("next_cls"),
        pl.col("t").shift(-1).over("ch").alias("next_t"),
    )
    tau_s = tau_minutes * 60.0
    ep = cp.filter(pl.col("cls") == "F").with_columns(
        pl.when(pl.col("prev_cls") == "O").then(pl.lit("strict")).otherwise(pl.lit("ambiguous")).alias("onset_type"),
        (pl.col("next_cls") == "O").fill_null(False).alias("exact"),
        pl.col("next_t").fill_null(pl.lit(data_end)).alias("end_t"),
    ).with_columns(
        (pl.col("end_t") - pl.col("t")).dt.total_seconds().cast(pl.Float64).alias("duration_s"),
    ).with_columns(
        pl.when(pl.col("exact") & (pl.col("duration_s") >= tau_s)).then(pl.lit(1, pl.Int8))
        .when(pl.col("exact")).then(pl.lit(0, pl.Int8))
        .otherwise(None).alias("sustained"),
    )
    return ep.select("ch", pl.col("t").alias("onset_t"), "onset_type", "exact", "end_t", "duration_s", "sustained")


# ---------------------------------------------------------------------------
# 4. Дневные агрегаты
# ---------------------------------------------------------------------------
COUNT_CLASSES = ["link_fault", "power_off", "power_on", "power_degraded", "device_off", "service",
                 "undefined_measurement", "object_summary", "security_mode", "flood", "event_alarm", "normal",
                 "numeric", "tech_value", "gas_out_of_range", "clock_1970", "dup_timestamp", "date_other",
                 "unknown_value", "empty_value"]


def daily_aggregates(pl: Any, ev: Any, sec: Any, tax: dict[str, Any]) -> Any:
    thr = tax["numeric"]["gas_threshold_percent"]
    low = tax["numeric"]["gas_band_low"]
    warn = tax["numeric"]["gas_warning_percent"]
    # порядок внутри группы = время; в одной секунде значения не упорядочены — берём детерминированно (по значению)
    ev = ev.sort("ch", "t", "num", nulls_last=False, maintain_order=True).with_columns(pl.col("t").dt.date().alias("day"))
    numeric = pl.col("cls") == "numeric"
    gas = numeric & pl.col("is_gas")
    agg = ev.group_by("ch", "day").agg(
        pl.len().alias("n_events"),
        pl.col("alarm").sum().alias("n_alarm"),
        *[(pl.col("cls") == c).sum().alias(f"n_{c}") for c in COUNT_CLASSES],
        pl.col("num").filter(numeric).mean().alias("num_mean"),
        pl.col("num").filter(numeric).min().alias("num_min"),
        pl.col("num").filter(numeric).max().alias("num_max"),
        pl.col("num").filter(numeric).std().alias("num_std"),
        pl.col("num").filter(numeric).last().alias("num_last"),
        pl.col("num").filter(gas).max().alias("gas_max"),
        pl.col("num").filter(gas).count().alias("gas_n"),
        (gas & (pl.col("num") >= low) & (pl.col("num") < thr)).sum().alias("gas_band_n"),
        (gas & (pl.col("num") >= thr)).sum().alias("gas_ge_thr_n"),
        (gas & (pl.col("num") >= warn)).sum().alias("gas_ge_warn_n"),
        pl.col("num").filter(gas).last().alias("gas_last"),
        # последняя газовая секунда суток содержит значения по обе стороны порога → сторона на конец D неизвестна (GPT 42)
        ((pl.col("num").filter(gas & (pl.col("t") == pl.col("t").filter(gas).max())).min() < thr)
         & (pl.col("num").filter(gas & (pl.col("t") == pl.col("t").filter(gas).max())).max() >= thr)).alias("gas_last_mixed"),
    )
    s = sec.with_columns(pl.col("t").dt.date().alias("day")).group_by("ch", "day").agg(
        (pl.col("n_values") > 1).sum().alias("n_multi_value_seconds"),
        (pl.col("link_cls") == "F").sum().alias("n_link_fault_seconds"),
        (pl.col("link_cls") == "A").sum().alias("n_link_ambiguous_seconds"),
        (pl.col("power_cls") == "F").sum().alias("n_power_off_seconds"),
    )
    return agg.join(s, on=["ch", "day"], how="left")


def gas_crossings(pl: Any, ev: Any, carry: Any, thr: float) -> tuple[Any, Any]:
    """Строгое пересечение порога на секундном уровне (секунда = множество значений без порядка, как для состояний):
    strict — все показания секунды ≥ thr, а все показания предыдущей секунды того же канала < thr;
    неоднозначно (cross_unknown) — в секунде есть значения по обе стороны порога, либо предыдущая секунда такая,
    либо предыдущего показания нет. Раньше показания одной секунды сортировались по значению, и пара 0 / 2,55
    в одну секунду давала ложный strict crossing (GPT 37).
    carry: последняя газовая секунда каждого канала из предыдущего года (ch, t, mn, mx)."""
    g = (ev.filter((pl.col("cls") == "numeric") & pl.col("is_gas"))
         .group_by("ch", "t").agg(pl.col("num").min().alias("mn"), pl.col("num").max().alias("mx")))
    both = pl.concat([carry.select("ch", "t", "mn", "mx").with_columns(pl.lit(True).alias("_carry")),
                      g.with_columns(pl.lit(False).alias("_carry"))], how="vertical").sort("ch", "t", "_carry",
                                                                                          descending=[False, False, True])
    pmn, pmx = pl.col("mn").shift(1).over("ch"), pl.col("mx").shift(1).over("ch")
    straddle = (pl.col("mn") < thr) & (pl.col("mx") >= thr)
    prev_straddle = (pmn < thr) & (pmx >= thr)
    x = both.with_columns(
        ((pl.col("mn") >= thr) & (pmx < thr)).fill_null(False).alias("cross"),
        ((pl.col("mx") >= thr) & (pmx.is_null() | straddle | ((pl.col("mn") >= thr) & prev_straddle.fill_null(False))))
        .fill_null(False).alias("cross_unknown_prev"))
    x = x.filter(~pl.col("_carry"))
    daily = x.with_columns(pl.col("t").dt.date().alias("day")).group_by("ch", "day").agg(
        pl.col("cross").sum().alias("gas_cross_n"), pl.col("cross_unknown_prev").sum().alias("gas_cross_unknown_n"))
    new_carry = (pl.concat([carry.select("ch", "t", "mn", "mx"), g], how="vertical").sort("ch", "t")
                 .group_by("ch", maintain_order=True).agg(pl.col("t").last(), pl.col("mn").last(), pl.col("mx").last()))
    return daily, new_carry


# ---------------------------------------------------------------------------
# 5. Сборка панели
# ---------------------------------------------------------------------------
def _shift_days(pl: Any, frame: Any, cols: list[str], offset: int, suffix: str) -> Any:
    """Факты дня X → строки D = X − offset."""
    return frame.select("ch", (pl.col("day") - pl.duration(days=offset)).alias("day"),
                        *[pl.col(c).alias(f"{c}{suffix}") for c in cols])


def build_panel_from_parts(pl: Any, daily: Any, link_cp: Any, power_cp: Any, gas_daily: Any,
                           meta: Any, data_end: datetime, config: dict[str, Any]) -> tuple[Any, dict[str, Any]]:
    tau = float(config["tau_minutes"])
    L = int(config["confirm_hours"])
    lookback = int(config["active_lookback_days"])
    data_end_day = data_end.date()

    link_ep = episodes(pl, link_cp, data_end, tau)
    power_ep = episodes(pl, power_cp, data_end, tau)

    daily = daily.join(gas_daily, on=["ch", "day"], how="left").with_columns(
        pl.col("gas_cross_n").fill_null(0), pl.col("gas_cross_unknown_n").fill_null(0))

    # грид: (канал, D), если канал слал события в [D-lookback+1; D]
    grid = (daily.select("ch", "day")
            .with_columns(pl.int_ranges(0, lookback).alias("_k")).explode("_k")
            .with_columns((pl.col("day") + pl.duration(days=pl.col("_k"))).alias("day"))
            .filter(pl.col("day") <= pl.lit(data_end_day))
            .select("ch", "day").unique().sort("ch", "day"))

    count_cols = [c for c in daily.columns if c.startswith("n_") or c in ("gas_n", "gas_band_n", "gas_ge_thr_n",
                                                                               "gas_ge_warn_n", "gas_cross_n")]
    panel = grid.join(daily, on=["ch", "day"], how="left").with_columns(
        [pl.col(c).fill_null(0) for c in count_cols]).sort("ch", "day")

    # эпизоды по дням: onset-день (для меток и признаков) и день завершения (для признаков «прошлых эпизодов»)
    def per_day(ep: Any, prefix: str, cp: Any) -> tuple[Any, Any, Any]:
        on = ep.with_columns(pl.col("onset_t").dt.date().alias("day")).group_by("ch", "day").agg(
            (pl.col("onset_type") == "strict").sum().alias(f"{prefix}_onset_strict"),
            (pl.col("onset_type") == "ambiguous").sum().alias(f"{prefix}_onset_ambiguous"),
            ((pl.col("onset_type") == "strict") & (pl.col("sustained") == 1)).sum().alias(f"{prefix}_onset_sustained"),
            ((pl.col("onset_type") == "strict") & (pl.col("sustained") == 0)).sum().alias(f"{prefix}_onset_transient"),
            ((pl.col("onset_type") == "strict") & pl.col("sustained").is_null()).sum().alias(f"{prefix}_onset_censored"),
        )
        done = ep.filter(pl.col("exact")).with_columns(pl.col("end_t").dt.date().alias("day")).group_by("ch", "day").agg(
            (pl.col("sustained") == 1).sum().alias(f"{prefix}_completed_long"),
            (pl.col("sustained") == 0).sum().alias(f"{prefix}_completed_short"),
        )
        # неоднозначные секунды (A-блоки): переход мог быть скрыт — окно с таким блоком не отрицательное (GPT 14, C2/C12)
        amb = cp.filter(pl.col("cls") == "A").with_columns(pl.col("t").dt.date().alias("day")).group_by("ch", "day").agg(
            pl.len().cast(pl.Int64).alias(f"{prefix}_ambiguous_block"))
        return on, done, amb

    link_on, link_done, link_amb = per_day(link_ep, "link", link_cp)
    power_on, power_done, power_amb = per_day(power_ep, "power", power_cp)
    for part in (link_on, link_done, power_on, power_done, link_amb, power_amb):
        cols = [c for c in part.columns if c not in ("ch", "day")]
        panel = panel.join(part, on=["ch", "day"], how="left").with_columns([pl.col(c).fill_null(0) for c in cols])

    # скользящие признаки (только прошлое, включая D)
    roll_src = ["n_events", "n_alarm", "n_link_fault", "n_power_off", "n_service", "n_device_off",
                "n_undefined_measurement", "n_tech_value", "n_gas_out_of_range", "n_clock_1970", "n_dup_timestamp",
                "n_date_other", "n_unknown_value", "n_flood", "n_event_alarm", "n_numeric", "n_multi_value_seconds",
                "n_link_fault_seconds", "n_link_ambiguous_seconds", "n_power_off_seconds",
                "link_onset_strict", "link_onset_ambiguous", "power_onset_strict",
                "link_completed_long", "link_completed_short", "power_completed_long", "power_completed_short",
                "gas_n", "gas_band_n", "gas_ge_thr_n", "gas_ge_warn_n", "gas_cross_n"]
    panel = panel.with_columns((pl.col("n_events") > 0).cast(pl.Int32).alias("active_day"),
                               (pl.col("n_tech_value") + pl.col("n_gas_out_of_range") + pl.col("n_undefined_measurement"))
                               .alias("n_tech_like"),
                               (pl.col("n_numeric") + pl.col("n_undefined_measurement") + pl.col("n_tech_value")
                                + pl.col("n_gas_out_of_range")).alias("n_measurement"))
    roll_exprs = []
    for w in config["window_offsets_days"]:
        for c in roll_src + ["active_day", "n_tech_like", "n_measurement"]:
            # polars < 1.3x не принимает null во входе rolling_*_by: счётчики без событий = 0
            roll_exprs.append(pl.col(c).fill_null(0).rolling_sum_by("day", window_size=f"{w}d", closed="right")
                              .over("ch").alias(f"f_{c}_{w}d"))
        # максимум газа: дни без газовых чисел → −∞ во входе, окно без чисел → null на выходе
        gmax = pl.col("gas_max").fill_null(float("-inf")).rolling_max_by("day", window_size=f"{w}d", closed="right").over("ch")
        roll_exprs.append(pl.when(gmax == float("-inf")).then(None).otherwise(gmax).alias(f"f_gas_max_{w}d"))
    panel = panel.sort("ch", "day").with_columns(roll_exprs)
    # последнее газовое/числовое значение на конец D (forward fill по грид-строкам канала)
    panel = panel.with_columns(pl.col("gas_last").forward_fill().over("ch").alias("f_gas_last"),
                               pl.col("num_last").forward_fill().over("ch").alias("f_num_last"),
                               pl.when(pl.col("gas_last").is_not_null()).then(pl.col("gas_last_mixed").fill_null(False))
                               .forward_fill().over("ch").fill_null(False).alias("_gas_last_mixed"))

    # состояния на конец D (as-of по точкам смены)
    cutoff = (pl.col("day").cast(pl.Datetime("us")) + pl.duration(days=1) - pl.duration(microseconds=1))
    panel = panel.with_columns(cutoff.alias("_cutoff_t")).sort("_cutoff_t")
    for name, cp in (("link", link_cp), ("power", power_cp)):
        c = cp.select("ch", pl.col("t").alias(f"_{name}_t"), pl.col("cls").alias(f"{name}_state")).sort(f"_{name}_t")
        panel = panel.join_asof(c, left_on="_cutoff_t", right_on=f"_{name}_t", by="ch", strategy="backward")
        panel = panel.with_columns(
            ((pl.col("_cutoff_t") - pl.col(f"_{name}_t")).dt.total_seconds() / 86400.0).alias(f"f_{name}_state_age_days"))
        last_on = (link_ep if name == "link" else power_ep).filter(pl.col("onset_type") == "strict").select(
            "ch", pl.col("onset_t").alias(f"_{name}_last_onset")).sort(f"_{name}_last_onset")
        panel = panel.join_asof(last_on, left_on="_cutoff_t", right_on=f"_{name}_last_onset", by="ch",
                                strategy="backward").with_columns(
            ((pl.col("_cutoff_t") - pl.col(f"_{name}_last_onset")).dt.total_seconds() / 86400.0)
            .alias(f"f_days_since_{name}_onset"))
    panel = panel.sort("ch", "day")

    # факты будущих дней для меток
    fact_cols = ["n_events", "link_onset_strict", "link_onset_ambiguous", "link_onset_sustained", "link_onset_censored",
                 "link_onset_transient", "power_onset_strict", "power_onset_ambiguous", "power_onset_sustained",
                 "power_onset_censored", "power_onset_transient", "link_ambiguous_block", "power_ambiguous_block",
                 "n_tech_like", "n_measurement", "gas_n",
                 "gas_cross_n", "gas_cross_unknown_n"]
    # каждый наблюдённый день есть в гриде (k = 0), поэтому факты любого дня с событиями — в panel
    facts = panel.select("ch", "day", *fact_cols)
    d1 = _shift_days(pl, facts, ["link_onset_strict", "link_onset_ambiguous", "power_onset_strict",
                                 "power_onset_ambiguous", "gas_cross_n"], 1, "_d1")
    d2 = _shift_days(pl, facts, fact_cols, 2, "_d2")
    d3 = _shift_days(pl, facts, ["n_events"], 3, "_d3")
    panel = panel.join(d1, on=["ch", "day"], how="left").join(d2, on=["ch", "day"], how="left") \
                 .join(d3, on=["ch", "day"], how="left")
    fut_cols = [c for c in panel.columns if c.endswith(("_d1", "_d2", "_d3"))]
    panel = panel.with_columns([pl.col(c).fill_null(0) for c in fut_cols])

    in_data = (pl.col("day") + pl.duration(days=3)) <= pl.lit(data_end_day)  # окно + L внутри данных
    observed = in_data & ((pl.col("n_events_d2") + pl.col("n_events_d3")) > 0)

    def onset_target(prefix: str, state_col: str, sustained: bool) -> tuple[Any, Any]:
        strict = pl.col(f"{prefix}_onset_strict_d2")
        amb = pl.col(f"{prefix}_onset_ambiguous_d2") + pl.col(f"{prefix}_ambiguous_block_d2")
        eligible = pl.col(state_col) == "O"
        if not sustained:
            label = (pl.when(~eligible.fill_null(False)).then(None)
                     .when(~in_data).then(None)
                     .when(strict > 0).then(pl.lit(1))
                     .when(amb > 0).then(None)
                     .when(observed).then(pl.lit(0))
                     .otherwise(None))
            reason = (pl.when(pl.col(state_col).is_null()).then(pl.lit("no_state_history"))
                      .when(~eligible).then(pl.lit("state_not_ok_at_D"))
                      .when(~in_data).then(pl.lit("window_beyond_data"))
                      .when(strict > 0).then(pl.lit("positive"))
                      .when(amb > 0).then(pl.lit("ambiguous_onset"))
                      .when(observed).then(pl.lit("negative"))
                      .otherwise(pl.lit("unobserved_window")))
        else:
            sus, cen, tra = (pl.col(f"{prefix}_onset_sustained_d2"), pl.col(f"{prefix}_onset_censored_d2"),
                             pl.col(f"{prefix}_onset_transient_d2"))
            label = (pl.when(~eligible.fill_null(False)).then(None)
                     .when(~in_data).then(None)
                     .when(sus > 0).then(pl.lit(1))
                     .when(cen > 0).then(None)
                     .when(amb > 0).then(None)
                     .when(observed).then(pl.lit(0))  # включая только кратковременные (transient) onset
                     .otherwise(None))
            reason = (pl.when(pl.col(state_col).is_null()).then(pl.lit("no_state_history"))
                      .when(~eligible).then(pl.lit("state_not_ok_at_D"))
                      .when(~in_data).then(pl.lit("window_beyond_data"))
                      .when(sus > 0).then(pl.lit("positive"))
                      .when(cen > 0).then(pl.lit("censored_duration"))
                      .when(amb > 0).then(pl.lit("ambiguous_onset"))
                      .when(observed & (tra > 0)).then(pl.lit("negative_transient_only"))
                      .when(observed).then(pl.lit("negative"))
                      .otherwise(pl.lit("unobserved_window")))
        return label.cast(pl.Int8), reason

    targets = {}
    reasons = {}
    for name, prefix, state, sus in (("target_t2a_link_onset", "link", "link_state", False),
                                     ("target_t2b_link_sustained", "link", "link_state", True),
                                     ("target_t2c_power_onset", "power", "power_state", False),
                                     ("target_t2c_power_sustained", "power", "power_state", True)):
        lab, rea = onset_target(prefix, state, sus)
        targets[name], reasons[name] = lab, rea

    # T1a: технические/диапазонные значения после ≥ tech_clean_days чистых суток
    clean = int(config["tech_clean_days"])
    hist = int(config["numeric_history_days"])
    panel = panel.with_columns(
        pl.col("n_tech_like").fill_null(0).rolling_sum_by("day", window_size=f"{clean}d", closed="right").over("ch").alias("_tech_recent"),
        pl.col("n_measurement").fill_null(0).rolling_sum_by("day", window_size=f"{hist}d", closed="right").over("ch").alias("_meas_hist"),
    )
    t1_elig = (pl.col("_meas_hist") > 0) & (pl.col("_tech_recent") == 0)
    targets["target_t1a_tech_value"] = (pl.when(~t1_elig | ~in_data).then(None)
                                        .when(pl.col("n_tech_like_d2") > 0).then(pl.lit(1))
                                        .when(pl.col("n_measurement_d2") > 0).then(pl.lit(0))
                                        .otherwise(None)).cast(pl.Int8)
    reasons["target_t1a_tech_value"] = (pl.when(pl.col("_meas_hist") == 0).then(pl.lit("no_measurements_30d"))
                                        .when(pl.col("_tech_recent") > 0).then(pl.lit("recent_tech_value"))
                                        .when(~in_data).then(pl.lit("window_beyond_data"))
                                        .when(pl.col("n_tech_like_d2") > 0).then(pl.lit("positive"))
                                        .when(pl.col("n_measurement_d2") > 0).then(pl.lit("negative"))
                                        .otherwise(pl.lit("unobserved_window")))

    # T4: газ — строгое пересечение 1 % в окне; канал газовый, последнее показание на конец D < порога
    thr = config["gas_threshold_percent"]
    gas_elig = (pl.col("f_gas_n_7d") > 0) & (pl.col("f_gas_last") < thr) & ~pl.col("_gas_last_mixed")
    targets["target_t4_gas_cross"] = (pl.when(~gas_elig.fill_null(False) | ~in_data).then(None)
                                      .when(pl.col("gas_cross_n_d2") > 0).then(pl.lit(1))
                                      .when(pl.col("gas_cross_unknown_n_d2") > 0).then(None)
                                      .when(pl.col("gas_n_d2") > 0).then(pl.lit(0))
                                      .otherwise(None)).cast(pl.Int8)
    reasons["target_t4_gas_cross"] = (pl.when(pl.col("f_gas_n_7d").fill_null(0) == 0).then(pl.lit("not_gas_stream"))
                                      .when(pl.col("_gas_last_mixed")).then(pl.lit("ambiguous_last_reading_at_D"))
                                      .when(pl.col("f_gas_last") >= thr).then(pl.lit("above_threshold_at_D"))
                                      .when(~in_data).then(pl.lit("window_beyond_data"))
                                      .when(pl.col("gas_cross_n_d2") > 0).then(pl.lit("positive"))
                                      .when(pl.col("gas_cross_unknown_n_d2") > 0).then(pl.lit("unknown_previous_reading"))
                                      .when(pl.col("gas_n_d2") > 0).then(pl.lit("negative"))
                                      .otherwise(pl.lit("unobserved_window")))

    panel = panel.with_columns(
        **targets, **{f"reason_{k[len('target_'):]}": v for k, v in reasons.items()},
        competing_link_onset_d1=(pl.col("link_onset_strict_d1") + pl.col("link_onset_ambiguous_d1") > 0),
        competing_power_onset_d1=(pl.col("power_onset_strict_d1") + pl.col("power_onset_ambiguous_d1") > 0),
        competing_gas_cross_d1=(pl.col("gas_cross_n_d1") > 0),
    )

    # мета: тип, объект, псевдоним ключей; календарь
    panel = panel.join(meta, on="ch", how="left").with_columns(
        pl.col("day").alias("d_cutoff_date"),
        (pl.col("day") + pl.duration(days=2)).alias("d_target_start_date"),
        (pl.col("day") + pl.duration(days=3)).alias("d_target_end_date_exclusive"),
        (pl.col("day") + pl.duration(days=3) + pl.duration(hours=L)).alias("d_label_decision_end"),
        pl.col("day").dt.year().alias("d_year"),
        pl.col("day").dt.weekday().alias("d_weekday"),
        pl.col("day").dt.month().alias("d_month"),
    )
    keys = key_map(pl, panel["ch"].unique().to_list(), "d_channel_key")
    objs = key_map(pl, panel["ид_объект"].drop_nulls().unique().to_list(), "d_object_key")
    panel = (panel.join(keys, left_on="ch", right_on="raw", how="left")
             .join(objs, left_on="ид_объект", right_on="raw", how="left"))

    feature_cols = [c for c in panel.columns if c.startswith("f_")] + BASE_FEATURES
    target_cols = list(targets)
    assert target_cols == TARGETS
    keep = (["d_channel_key", "d_object_key", "d_cutoff_date", "d_target_start_date", "d_target_end_date_exclusive",
             "d_label_decision_end", "d_year", "тип_датчика", "тип_инж_системы"] + feature_cols + target_cols
            + [f"reason_{k[len('target_'):]}" for k in target_cols]
            + ["competing_link_onset_d1", "competing_power_onset_d1", "competing_gas_cross_d1"])
    out = panel.select(keep).sort("d_channel_key", "d_cutoff_date")

    durations = pl.concat([
        e.select(pl.lit(n).alias("kind"), "onset_type", "exact", "duration_s", "sustained")
        for n, e in (("link", link_ep), ("power", power_ep))], how="vertical")
    return out, {"rows": out.height, "channels": out["d_channel_key"].n_unique(), "_episode_durations": durations}


# ---------------------------------------------------------------------------
# 6. Аудит (только агрегаты)
# ---------------------------------------------------------------------------
def final_audit(pl: Any, panel_lf: Any, durations: Any) -> dict[str, Any]:
    def ep_stats(kind: str) -> dict[str, Any]:
        ep = durations.filter(pl.col("kind") == kind)
        s = ep.filter(pl.col("onset_type") == "strict")
        dur = s.filter(pl.col("exact"))["duration_s"]
        return {
            "episodes": ep.height, "strict_onsets": s.height,
            "ambiguous_onsets": int((ep["onset_type"] == "ambiguous").sum()),
            "strict_exact": dur.len(), "strict_censored": int((~s["exact"]).sum()),
            "exact_duration_s_quantiles": ({str(q): float(dur.quantile(q)) for q in (0.1, 0.5, 0.9, 0.99)} if dur.len() else None),
            "sustained_1": int((s["sustained"] == 1).sum()), "sustained_0": int((s["sustained"] == 0).sum()),
            "sustained_null": int(s["sustained"].is_null().sum()),
        }

    head = panel_lf.select(pl.len().alias("rows"), pl.col("d_channel_key").n_unique().alias("channels"),
                           pl.col("d_cutoff_date").min().alias("dmin"), pl.col("d_cutoff_date").max().alias("dmax")).collect()
    out: dict[str, Any] = {"rows": int(head["rows"][0]), "channels": int(head["channels"][0]),
                           "date_min": str(head["dmin"][0]), "date_max": str(head["dmax"][0]),
                           "link_episodes": ep_stats("link"), "power_episodes": ep_stats("power"), "targets": {}}
    for t in TARGETS:
        r = f"reason_{t[len('target_'):]}"
        g = panel_lf.group_by(r).agg(pl.len()).collect()
        by_year = panel_lf.group_by("d_year").agg((pl.col(t) == 1).sum().alias("pos"), (pl.col(t) == 0).sum().alias("neg"),
                                                  pl.col(t).null_count().alias("null")).sort("d_year").collect()
        reasons = {k: int(v) for k, v in g.iter_rows()}
        out["targets"][t] = {
            "positive": reasons.get("positive", 0),
            "negative": reasons.get("negative", 0) + reasons.get("negative_transient_only", 0),
            "null_reasons": {k: v for k, v in reasons.items() if k not in ("positive", "negative", "negative_transient_only")},
            "negative_transient_only": reasons.get("negative_transient_only", 0),
            "by_year": {str(y): {"pos": int(p), "neg": int(n), "null": int(nu)} for y, p, n, nu in by_year.iter_rows()},
        }
    return out


# ---------------------------------------------------------------------------
# 7. Поток по годам (для Kaggle): сырой журнал → части → панель
# ---------------------------------------------------------------------------
def read_journal(pl: Any, path: Path) -> Any:
    schema = {c: pl.String for c in EVENT_COLUMNS}
    return pl.read_csv(path, schema_overrides=schema, infer_schema_length=0, encoding="utf8-lossy",
                       columns=EVENT_COLUMNS).filter(pl.col("ид_события") != "ид_события")


def read_catalogue(pl: Any, path: Path) -> Any:
    cat = pl.read_csv(path, infer_schema_length=0, encoding="utf8-lossy")
    return cat.select([c for c in CATALOGUE_COLUMNS if c in cat.columns]).unique("ид_канала_данных", keep="last")


def process_year(pl: Any, raw: Any, tax: dict[str, Any], catalogue: Any, gas_carry: Any) -> dict[str, Any]:
    ev = classify_events(pl, raw, tax, catalogue)
    sec = second_table(pl, ev, tax)
    daily = daily_aggregates(pl, ev, sec, tax)
    gas_daily, gas_carry = gas_crossings(pl, ev, gas_carry, tax["numeric"]["gas_threshold_percent"])
    meta = ev.group_by("ch").agg(pl.col("тип_датчика").drop_nulls().last(), pl.col("тип_инж_системы").drop_nulls().last(),
                                 pl.col("ид_объект").drop_nulls().last())
    class_counts = {k: int(v) for k, v in ev.group_by("cls").len().iter_rows()}
    streams = tax["stream_mapping"]["streams"]
    mapped = {c for st in streams.values() for key in ("fault", "ok", "unknown") for c in st[key]}
    stream_audit = {}
    for name in streams:
        g = sec.group_by(f"{name}_cls", f"{name}_a_cause").len()
        stream_audit[name] = {f"{st or 'none'}{'_' + cause if cause else ''}": int(n) for st, cause, n in g.iter_rows()}
    stream_audit["events_in_no_stream_by_class"] = {k: v for k, v in class_counts.items() if k not in mapped}
    # «Неопределен»: сколько каналов и какие классы соседствуют (агрегаты, порядок внутри секунды не используется)
    evs = ev.select("ch", "t", "cls").sort("ch", "t")
    evs = evs.with_columns(pl.col("cls").shift(1).over("ch").alias("prev"), pl.col("cls").shift(-1).over("ch").alias("next"))
    svc = evs.filter(pl.col("cls") == "service")
    service_audit = {"events": svc.height, "channels": svc["ch"].n_unique(),
                     "prev_class": {str(k): int(v) for k, v in svc.group_by("prev").len().sort("len", descending=True).head(10).iter_rows()},
                     "next_class": {str(k): int(v) for k, v in svc.group_by("next").len().sort("len", descending=True).head(10).iter_rows()}}
    unknown_values = {k: int(v) for k, v in ev.filter(pl.col("cls") == "unknown_value").group_by("v").len()
                      .sort("len", descending=True).head(30).iter_rows()}
    return {
        "daily": daily, "link_cp": change_points(pl, sec, "link_cls"), "power_cp": change_points(pl, sec, "power_cls"),
        "gas_daily": gas_daily, "gas_carry": gas_carry, "meta": meta, "t_max": ev["t"].max(),
        "audit": {"events": ev.height, "channels": ev["ch"].n_unique(), "seconds": sec.height,
                  "multi_value_seconds": int((sec["n_values"] > 1).sum()),
                  "link_fault_seconds": int((sec["link_cls"] == "F").sum()),
                  "link_fault_seconds_with_other_value": int(((sec["has_fault"]) & (sec["n_values"] > 1)).sum()),
                  "link_ambiguous_seconds": int((sec["link_cls"] == "A").sum()),
                  "class_counts": class_counts, "unknown_values_top30": unknown_values,
                  "stream_mapping": stream_audit, "service_value": service_audit},
    }


def empty_gas_carry(pl: Any) -> Any:
    return pl.DataFrame(schema={"ch": pl.String, "t": pl.Datetime("us"), "mn": pl.Float64, "mx": pl.Float64})


def journal_max_time(pl: Any, path: Path) -> datetime:
    """Лёгкий предпроход: последний момент в файле (для общей границы данных)."""
    t = pl.concat_str(["дата", "время"], separator=" ").str.strptime(pl.Datetime("us"), "%Y-%m-%d %H:%M:%S", strict=False)
    return _scan_source(pl, path).select(t.max()).collect().item()


def _scan_source(pl: Any, path: Path) -> Any:
    path = Path(path)
    if path.suffix == ".parquet":
        return pl.scan_parquet(path)
    schema = {c: pl.String for c in EVENT_COLUMNS}
    return pl.scan_csv(path, schema_overrides=schema, infer_schema_length=0, encoding="utf8-lossy")


def scan_bucket(pl: Any, path: Path, bucket: int, n_buckets: int, channel_filter: Any = None) -> Any:
    lf = _scan_source(pl, path).select(EVENT_COLUMNS).filter(pl.col("ид_события") != "ид_события")
    if n_buckets > 1:
        lf = lf.filter(pl.col("ид_канала_данных").hash(seed=20260923) % n_buckets == bucket)
    if channel_filter is not None:
        lf = lf.filter(channel_filter)
    return lf.collect()


def _merge_audits(parts: list[dict[str, Any]]) -> dict[str, Any]:
    out: dict[str, Any] = {}
    for a in parts:
        for k, v in a.items():
            if isinstance(v, bool) or v is None:
                out.setdefault(k, v)
            elif isinstance(v, (int, float)):
                out[k] = out.get(k, 0) + v
            elif isinstance(v, dict):
                out[k] = _merge_audits([out.get(k, {}), v])
            else:
                out.setdefault(k, v)
    return out


def build_event_panel(pl: Any, journal_paths: list[Path], catalogue_path: Path, taxonomy_path: Path,
                      config: dict[str, Any] | None = None, log: Any = print, n_buckets: int = 1,
                      out_dir: Path | None = None, channel_filter: Any = None,
                      data_end: datetime | None = None) -> tuple[Any, dict[str, Any]]:
    """Собирает панель по группам каналов (каналы независимы). При out_dir пишет части parquet и
    возвращает (None, audit); иначе возвращает панель в памяти."""
    cfg = {**DEFAULT_CONFIG, **(config or {})}
    tax = load_taxonomy(taxonomy_path)
    cfg["gas_threshold_percent"] = tax["numeric"]["gas_threshold_percent"]
    if not cfg["service_as_unknown"]:
        link = tax["stream_mapping"]["streams"]["link"]
        link["unknown"] = [c for c in link["unknown"] if c != "service"]
    if cfg["power_ok_from_function"]:
        power = tax["stream_mapping"]["streams"]["power"]
        power["ok"] = list(dict.fromkeys(power["ok"] + ["normal", "event_alarm", "numeric"]))
    cfg["stream_mapping_version"] = tax["stream_mapping"]["version"]
    catalogue = read_catalogue(pl, catalogue_path)
    # data_end задаётся явно на runtime (конец дня D): события после D в сборку не попадают (см. channel_filter)
    t_max = data_end if data_end is not None else max(journal_max_time(pl, p) for p in journal_paths)
    panels, part_paths, per_file_parts, ep_parts = [], [], [], []
    for b in range(n_buckets):
        carry = empty_gas_carry(pl)
        parts: dict[str, list[Any]] = {"daily": [], "link_cp": [], "power_cp": [], "gas_daily": [], "meta": []}
        per_file = {}
        for path in journal_paths:
            raw = scan_bucket(pl, Path(path), b, n_buckets, channel_filter)
            if raw.is_empty():
                continue
            res = process_year(pl, raw, tax, catalogue, carry)
            carry = res["gas_carry"]
            for k in parts:
                parts[k].append(res[k])
            per_file[Path(path).name] = res["audit"]
            del raw, res
        if not parts["daily"]:
            continue
        daily = pl.concat(parts["daily"], how="vertical_relaxed")
        dup = daily.height - daily.select("ch", "day").unique().height
        if dup:
            raise ValueError(f"дневные агрегаты пересекаются между файлами: {dup} строк")
        link_cp = rle(pl, pl.concat(parts["link_cp"], how="vertical"))
        power_cp = rle(pl, pl.concat(parts["power_cp"], how="vertical"))
        gas_daily = pl.concat(parts["gas_daily"], how="vertical").group_by("ch", "day").agg(pl.all().sum())
        meta = pl.concat(parts["meta"], how="vertical_relaxed").group_by("ch").agg(pl.all().drop_nulls().last())
        panel, audit = build_panel_from_parts(pl, daily, link_cp, power_cp, gas_daily, meta, t_max, cfg)
        per_file_parts.append({"per_file": per_file})
        ep_parts.append(audit.pop("_episode_durations"))
        if out_dir is not None:
            path = Path(out_dir) / f"event_panel_v3_part{b:02d}.parquet"
            panel.write_parquet(path, compression="zstd")
            part_paths.append(path)
            del panel
        else:
            panels.append(panel)
        log(f"группа каналов {b + 1}/{n_buckets}: строк {audit['rows']:,}, каналов {audit['channels']:,}")
    if out_dir is not None:
        full = pl.scan_parquet(part_paths)
    else:
        full = pl.concat(panels, how="vertical_relaxed").lazy()
    audit = final_audit(pl, full, pl.concat(ep_parts, how="vertical"))
    audit["feature_columns"] = [c for c in full.collect_schema().names() if c.startswith("f_")] + BASE_FEATURES
    audit["target_columns"] = TARGETS
    audit.update({"schema_version": SCHEMA_VERSION, "config": cfg, "taxonomy_version": tax["version"],
                  "taxonomy_sha256": tax["_sha256"], "data_end": str(t_max), "n_buckets": n_buckets,
                  "per_file": _merge_audits(per_file_parts)["per_file"] if per_file_parts else {},
                  "part_files": [p.name for p in part_paths]})
    return (None if out_dir is not None else full.collect()), audit


def csv_to_parquet(pl: Any, csv_path: Path, out_path: Path) -> dict[str, Any]:
    """Сырой CSV → parquet только нужных колонок (все строки как строки, без интерпретации)."""
    schema = {c: pl.String for c in EVENT_COLUMNS}
    lf = pl.scan_csv(csv_path, schema_overrides=schema, infer_schema_length=0, encoding="utf8-lossy").select(EVENT_COLUMNS)
    lf.sink_parquet(out_path, compression="zstd")
    rows = pl.scan_parquet(out_path).select(pl.len()).collect().item()
    return {"rows": int(rows), "parquet_bytes": Path(out_path).stat().st_size}


# ---------------------------------------------------------------------------
# 8. Самопроверки на синтетике (для notebook)
# ---------------------------------------------------------------------------
def run_event_panel_self_tests(taxonomy_path: Path, work_dir: Path) -> dict[str, bool]:
    import polars as pl
    import random

    work_dir = Path(work_dir)
    work_dir.mkdir(parents=True, exist_ok=True)
    t0 = datetime(2024, 3, 1)

    def events(rows):
        return pl.DataFrame({"ид_события": [str(i) for i in range(len(rows))],
                             "ид_канала_данных": [r[0] for r in rows],
                             "дата": [r[1].strftime("%Y-%m-%d") for r in rows],
                             "время": [r[1].strftime("%H:%M:%S") for r in rows],
                             "тревожное": ["f"] * len(rows), "значение_датчика": [r[2] for r in rows]})

    def catalogue(chs, typ="Состояние насоса"):
        return pl.DataFrame({"ид_канала_данных": chs, "тип_инж_системы": ["x"] * len(chs),
                             "тип_датчика": [typ] * len(chs), "ид_объект": ["1"] * len(chs)})

    def build(name, rows, chs, **kw):
        d = work_dir / name
        d.mkdir(exist_ok=True)
        events(rows).write_csv(d / "j.csv")
        catalogue(chs).write_csv(d / "c.csv")
        return build_event_panel(pl, [d / "j.csv"], d / "c.csv", taxonomy_path, log=lambda *a: None, **kw)

    def row(panel, ch, day):
        r = panel.filter((pl.col("d_channel_key") == pseudo_key(ch)) & (pl.col("d_cutoff_date") == (t0 + timedelta(days=day)).date()))
        return r.row(0, named=True)

    report: dict[str, bool] = {}
    hb = [("A", t0 + timedelta(days=d, hours=12), "Норма") for d in range(12)]
    p, _ = build("onset", hb + [("A", t0 + timedelta(days=5, hours=13), "Неисправен"), ("A", t0 + timedelta(days=5, hours=14), "Норма")], ["A"])
    report["strict_onset_in_window_is_positive"] = row(p, "A", 3)["target_t2a_link_onset"] == 1
    report["onset_in_d1_is_competing_not_filtered"] = row(p, "A", 4)["target_t2a_link_onset"] == 0 and row(p, "A", 4)["competing_link_onset_d1"]
    t = t0 + timedelta(days=5, hours=13)
    p, _ = build("amb", hb + [("A", t, "Неисправен"), ("A", t, "Норма")], ["A"])
    r = row(p, "A", 3)  # конфликт в одной секунде → не onset и не 0, а null(ambiguous_onset)
    report["same_second_conflict_is_null"] = r["target_t2a_link_onset"] is None and r["reason_t2a_link_onset"] == "ambiguous_onset"
    rnd = random.Random(1)
    vals = ["Норма", "Неисправен", "Обесточен", "Есть питание", "Выключен", "Неопределен", "0,05", "1,3", "-127"]
    rows = [(f"C{c}", t0 + timedelta(days=d, seconds=rnd.randint(0, 86399)), rnd.choice(vals))
            for c in range(8) for d in range(30) for _ in range(rnd.randint(0, 5))]
    chs = [f"C{c}" for c in range(8)]
    base, _ = build("rand1", rows, chs)
    out = work_dir / "rand_parts"
    out.mkdir(exist_ok=True)
    build("rand2", rows, chs, n_buckets=3, out_dir=out)
    parts = pl.read_parquet(sorted(out.glob("*.parquet")))
    key = ["d_channel_key", "d_cutoff_date"]
    report["channel_buckets_do_not_change_panel"] = base.sort(key).equals(parts.sort(key))
    report["no_raw_identifiers_in_panel"] = not any(c in base.write_csv() for c in ("C1,", "C2,"))
    return report


def label_changes(pl: Any, main: Any, variant: Any, targets: list[str] | None = None) -> dict[str, dict[str, int]]:
    """Сколько меток меняется между основной и sensitivity-версией (только агрегаты, по ключу строки)."""
    k = ["d_channel_key", "d_cutoff_date"]
    out: dict[str, dict[str, int]] = {}
    for t in targets or TARGETS:
        j = main.select(k + [t]).join(variant.select(k + [t]), on=k, how="full", suffix="_v", coalesce=True)
        a = pl.col(t).cast(pl.String).fill_null("null")
        b = pl.col(f"{t}_v").cast(pl.String).fill_null("null")
        g = j.with_columns(a.alias("a"), b.alias("b")).filter(pl.col("a") != pl.col("b")).group_by("a", "b").len()
        out[t] = {f"{x}->{y}": int(n) for x, y, n in g.sort("a", "b").iter_rows()}
    return out


## Встроенная обвязка Kaggle: поиск входов, 7z → parquet, manifest


In [ ]:
"""Kaggle-обвязка notebook 14: сырые журналы → событийная панель v3 (части parquet) + manifest/аудит/сводка.

Шаги:
1. поиск входов в /kaggle/input: ext-journal-YYYY.{7z,csv,parquet} и справочник_каналов_датчиков.csv
   (имена сравниваются после Unicode-NFC: загрузка с macOS даёт NFD-кириллицу);
2. распаковка 7z по одному архиву → parquet только нужных колонок (все значения строками) → CSV удаляется;
3. перераскладка по календарным годам события (файл года может содержать события соседнего года;
   иначе дневные агрегаты пересеклись бы между файлами) и удаление точных дублей строк (счётчик в аудите);
4. сборка панели по группам каналов (каналы независимы) → event_panel_v3/event_panel_v3_partNN.parquet;
5. manifest с SHA-256 частей и входов, аудит и сводка — только агрегаты, без идентификаторов.
"""

from __future__ import annotations

import hashlib
import json
import os
import re
import shutil
import subprocess
import sys
import time
import unicodedata
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

# (импорт event_panel_v3 не нужен: модуль выполнен ячейкой выше)

JOURNAL_RE = re.compile(r"^ext-journal-(\d{4})\.(7z|csv|parquet)$")
CATALOGUE_NAME = "справочник_каналов_датчиков.csv"
SOURCE_PRIORITY = {"parquet": 0, "csv": 1, "7z": 2}
OUTPUT_FILES = ("panel_manifest_v3.json", "audit_event_panel_v3.json", "summary_event_panel_v3_ru.md")


def make_config_v3(mode: str, overrides: dict[str, Any] | None = None) -> dict[str, Any]:
    if mode not in ("FULL", "SMOKE"):
        raise ValueError("MODE должен быть FULL или SMOKE")
    cfg: dict[str, Any] = {
        "mode": mode,
        "input_dir": os.environ.get("LDT_KAGGLE_INPUT_DIR", "/kaggle/input"),
        "output_dir": os.environ.get("LDT_OUTPUT_DIR", "/kaggle/working"),
        "stage_dir": os.environ.get("LDT_STAGE_DIR", "/tmp/ldt_event_panel_v3_stage"),
        "taxonomy_path": None,
        "n_buckets": 4,
        "panel": {"tau_minutes": 5},
        "smoke_years": [2019, 2020],
        "smoke_channel_share": 20,   # SMOKE: ≈1/20 каналов
        "sensitivity_variants": ["service_neutral", "power_ok_from_function"],  # отдельные сборки, в выход — только агрегаты
        "keep_stage": False,
    }
    cfg.update(overrides or {})
    return cfg


def _nfc(name: str) -> str:
    return unicodedata.normalize("NFC", name)


def file_sha256(path: Path, chunk: int = 1 << 22) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while block := f.read(chunk):
            h.update(block)
    return h.hexdigest()


CATALOGUE_REQUIRED_COLUMNS = {"ид_канала_данных", "тип_датчика"}
YEAR_IN_PATH_RE = re.compile(r"ext-journal-(\d{4})")


def _csv_header(path: Path) -> set[str]:
    """Имена колонок первой строки CSV (для файлов, которые Kaggle переименовал: «справочник…» → «__.csv»)."""
    try:
        with open(path, "r", encoding="utf-8-sig", errors="replace") as f:
            line = f.readline()
    except OSError:
        return set()
    return {c.strip().strip('"') for c in line.rstrip("\r\n").split(",")}


def discover_sources(input_dir: Path) -> dict[str, Any]:
    """Один источник на год (parquet > csv > 7z) и один справочник каналов.

    Сначала по имени файла; для CSV с непонятным именем — по заголовку. Kaggle при загрузке заменяет кириллицу
    в имени файла на «_» (справочник становится «__.csv») и распаковывает архивы в папки ext-journal-YYYY/."""
    by_year: dict[int, list[tuple[int, Path]]] = {}
    catalogues: list[Path] = []
    for p in sorted(Path(input_dir).rglob("*")):
        if not p.is_file():
            continue
        name = _nfc(p.name)
        m = JOURNAL_RE.match(name)
        if m:
            by_year.setdefault(int(m.group(1)), []).append((SOURCE_PRIORITY[m.group(2)], p))
            continue
        if name == CATALOGUE_NAME:
            catalogues.insert(0, p)
            continue
        if p.suffix.lower() != ".csv":
            continue
        header = _csv_header(p)
        if set(EVENT_COLUMNS) <= header:
            ym = YEAR_IN_PATH_RE.search(_nfc(str(p)))
            if ym:  # журнал с неожиданным именем файла, год — из пути (папка ext-journal-YYYY)
                by_year.setdefault(int(ym.group(1)), []).append((SOURCE_PRIORITY["csv"] + 5, p))
        elif CATALOGUE_REQUIRED_COLUMNS <= header and "значение_датчика" not in header:
            catalogues.append(p)
    journals = {y: sorted(v)[0][1] for y, v in sorted(by_year.items())}
    years = sorted(journals)
    missing = [y for y in range(years[0], years[-1] + 1) if y not in journals] if years else []
    return {"journals": journals, "ignored_duplicates": sum(len(v) - 1 for v in by_year.values()),
            "catalogue": catalogues[0] if catalogues else None, "catalogue_candidates": len(catalogues),
            "missing_years": missing}


def describe_sources(src: dict[str, Any]) -> list[str]:
    """Строки для preflight-ячейки: что найдено и чего не хватает (без содержимого файлов)."""
    lines = [f"журналы: {', '.join(str(y) for y in src['journals']) or 'не найдены'}"]
    for y, p in src["journals"].items():
        lines.append(f"  {y}: {p.parent.name}/{p.name} ({p.stat().st_size / 2**30:.2f} ГБ)")
    cat = src["catalogue"]
    lines.append(f"справочник каналов: {cat.name if cat else 'НЕ НАЙДЕН'}"
                 + (f" (кандидатов: {src['catalogue_candidates']})" if src["catalogue_candidates"] > 1 else ""))
    if src["missing_years"]:
        lines.append("ВНИМАНИЕ: нет журналов за " + ", ".join(map(str, src["missing_years"]))
                     + " — фолды ноутбука 15 обучаются на 2019–2020 и 2022+; без пропущенного года train неполный")
    return lines


def extract_7z(archive: Path, dest: Path, log: Any = print) -> list[Path]:
    dest.mkdir(parents=True, exist_ok=True)
    exe = next((shutil.which(x) for x in ("7z", "7za", "7zr") if shutil.which(x)), None)
    if exe:
        subprocess.run([exe, "x", "-y", f"-o{dest}", str(archive)], check=True,
                       stdout=subprocess.DEVNULL, stderr=subprocess.PIPE)
    else:
        try:
            import py7zr  # type: ignore
        except ImportError:
            log("7z не найден — ставлю py7zr (нужен Internet в Settings ноутбука)")
            try:
                subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "py7zr"])
                import py7zr  # type: ignore
            except Exception as exc:  # noqa: BLE001
                raise RuntimeError("Нет распаковщика 7z: включите Internet в Settings или загрузите в dataset "
                                   "распакованные ext-journal-YYYY.csv") from exc
        with py7zr.SevenZipFile(archive, "r") as z:
            z.extractall(dest)
    return sorted(p for p in dest.rglob("*") if p.is_file() and p.suffix.lower() == ".csv")


def stage_journals(pl: Any, journals: dict[int, Path], stage_dir: Path, years: list[int] | None,
                   log: Any = print) -> tuple[list[Path], dict[str, Any]]:
    """Источники → parquet → файлы по календарному году события. Возвращает (файлы годов, аудит)."""
    raw_dir, tmp_dir, year_dir = stage_dir / "raw_parquet", stage_dir / "extract", stage_dir / "by_year"
    for d in (raw_dir, year_dir):
        shutil.rmtree(d, ignore_errors=True)
        d.mkdir(parents=True)
    audit: dict[str, Any] = {"sources": [], "years": {}}
    raw_files: list[Path] = []
    for year, src in journals.items():
        if years is not None and year not in years:
            continue
        t0 = time.time()
        entry = {"name": _nfc(src.name), "bytes": src.stat().st_size, "sha256": file_sha256(src), "rows": 0}
        if src.suffix == ".parquet":
            raw_files.append(src)
            entry["rows"] = int(pl.scan_parquet(src).select(pl.len()).collect().item())
        else:
            csvs = [src] if src.suffix == ".csv" else extract_7z(src, tmp_dir, log)
            if not csvs:
                raise RuntimeError(f"в архиве {entry['name']} нет CSV")
            for i, csv in enumerate(csvs):
                out = raw_dir / f"{src.stem}_{i}.parquet"
                entry["rows"] += csv_to_parquet(pl, csv, out)["rows"]
                raw_files.append(out)
            shutil.rmtree(tmp_dir, ignore_errors=True)
        audit["sources"].append(entry)
        log(f"{entry['name']}: {entry['rows']:,} строк → parquet за {time.time() - t0:.0f} с")
    if not raw_files:
        return [], audit
    prefix = pl.col("дата").str.slice(0, 4)
    is_header = pl.col("ид_события") == "ид_события"
    # построчный баланс по каждому источнику: повторённые строки заголовка и строки без года тоже учитываются явно
    per_file = (pl.concat([pl.scan_parquet(f).select(pl.lit(f.name).alias("_file"), is_header.alias("_hdr"),
                                                     prefix.alias("y")) for f in raw_files])
                .group_by("_file").agg(pl.col("_hdr").sum().alias("header_rows"),
                                       (~pl.col("_hdr") & ~pl.col("y").fill_null("").str.contains(r"^\d{4}$"))
                                       .sum().alias("unparseable_year_rows")).collect())
    audit["raw_files"] = {name: {"header_rows": int(h), "unparseable_year_rows": int(u)}
                          for name, h, u in per_file.iter_rows()}
    counts = (pl.scan_parquet(raw_files).filter(~is_header)
              .group_by(prefix.alias("y")).agg(pl.len()).collect())
    bad = int(counts.filter(~pl.col("y").str.contains(r"^\d{4}$") | pl.col("y").is_null())["len"].sum())
    audit["rows_with_unparseable_year_excluded"] = bad
    audit["header_rows_excluded"] = int(per_file["header_rows"].sum())
    year_files = []
    for y in sorted(counts.filter(pl.col("y").str.contains(r"^\d{4}$"))["y"].to_list()):
        if years is not None and int(y) not in years:
            audit["years"][y] = {"skipped_outside_mode_years": int(counts.filter(pl.col("y") == y)["len"][0])}
            continue
        frames = []
        for f in raw_files:
            part = pl.scan_parquet(f).select(EVENT_COLUMNS).filter((prefix == y) & (pl.col("ид_события") != "ид_события")).collect()
            if part.height:
                frames.append((f, part))
        rows = pl.concat([p for _, p in frames], how="vertical")
        n_in = rows.height
        rows = rows.unique(maintain_order=True)
        # диагностика дублей: одинаковый ид_события при разном содержимом (такие строки НЕ удаляются)
        same_id_diff = rows.height - rows.select(pl.col("ид_события").n_unique()).item()
        from_other = sum(p.height for f, p in frames if f"-{y}" not in f.name)
        out = year_dir / f"journal_{y}.parquet"
        rows.write_parquet(out, compression="zstd")
        audit["years"][y] = {"rows_in": n_in, "rows": rows.height, "exact_duplicate_rows_dropped": n_in - rows.height,
                             "same_event_id_different_content_kept": int(same_id_diff),
                             "rows_from_other_year_files": from_other}
        year_files.append(out)
        del rows, frames
    src_rows = sum(e["rows"] for e in audit["sources"])
    staged = sum(v.get("rows", 0) for v in audit["years"].values())
    dups = sum(v.get("exact_duplicate_rows_dropped", 0) for v in audit["years"].values())
    skipped = sum(v.get("skipped_outside_mode_years", 0) for v in audit["years"].values())
    audit["row_balance"] = {"source_rows": src_rows, "staged_rows": staged, "exact_duplicates_dropped": dups,
                            "header_rows_excluded": audit["header_rows_excluded"],
                            "unparseable_year_rows_excluded": bad, "skipped_outside_mode_years": skipped,
                            "unexplained": src_rows - staged - dups - audit["header_rows_excluded"] - bad - skipped}
    if not os.environ.get("LDT_KEEP_RAW_STAGE"):
        shutil.rmtree(raw_dir, ignore_errors=True)
    return year_files, audit


def _truncate_unknown(audit: dict[str, Any]) -> None:
    """Неизвестные значения — только короткие строки (не выгружать длинный свободный текст)."""
    for fa in audit.get("per_file", {}).values():
        top = fa.get("unknown_values_top30")
        if top:
            fa["unknown_values_top30"] = {str(k)[:40]: v for k, v in top.items()}


def summary_markdown(manifest: dict[str, Any], audit: dict[str, Any]) -> str:
    L = [f"# Событийная панель v3 — сводка ({manifest['mode']})", ""]
    if manifest["mode"] == "SMOKE":
        L += ["**SMOKE:** ≈1/{} каналов, годы {}. Цифры не сравнимы с FULL.".format(
            manifest["config"]["smoke_channel_share"], manifest["config"]["smoke_years"]), ""]
    L += [f"Строк (канал × сутки D): **{audit['rows']:,}**, каналов: {audit['channels']:,}, "
          f"D от {audit['date_min']} до {audit['date_max']}; граница данных {audit['data_end']}.", "",
          f"Время сборки: {manifest['runtime_s']:.0f} с; частей: {len(manifest['parts'])}.", "",
          "## Цели", "",
          "| цель | 1 | 0 | null | доля 1 среди размеченных | главные причины null |", "|---|---|---|---|---|---|"]
    for t, s in audit["targets"].items():
        null = sum(s["null_reasons"].values())
        lab = s["positive"] + s["negative"]
        top = ", ".join(f"{k} {v:,}" for k, v in sorted(s["null_reasons"].items(), key=lambda kv: -kv[1])[:3])
        L.append(f"| `{t}` | {s['positive']:,} | {s['negative']:,} | {null:,} | "
                 f"{(s['positive'] / lab if lab else float('nan')):.4f} | {top} |")
    L += ["", "## Эпизоды (секундный уровень)", "", "| вид | эпизодов | strict onset | ambiguous | медиана длит., с | ≥ τ | < τ | цензура |",
          "|---|---|---|---|---|---|---|---|"]
    for kind in ("link_episodes", "power_episodes"):
        e = audit[kind]
        med = (e["exact_duration_s_quantiles"] or {}).get("0.5")
        L.append(f"| {kind.split('_')[0]} | {e['episodes']:,} | {e['strict_onsets']:,} | {e['ambiguous_onsets']:,} | "
                 f"{med if med is not None else '—'} | {e['sustained_1']:,} | {e['sustained_0']:,} | {e['sustained_null']:,} |")
    cls: dict[str, int] = {}
    ev = 0
    for fa in audit["per_file"].values():
        ev += fa.get("events", 0)
        for k, v in fa.get("class_counts", {}).items():
            cls[k] = cls.get(k, 0) + v
    L += ["", "## Классы значений", "", "| класс | событий | доля |", "|---|---|---|"]
    for k, v in sorted(cls.items(), key=lambda kv: -kv[1]):
        L.append(f"| {k} | {v:,} | {v / max(ev, 1):.4f} |")
    unk = cls.get("unknown_value", 0) / max(ev, 1)
    L += ["", "## Предупреждения", ""]
    warn = []
    if unk > 0.01:
        warn.append(f"- доля неизвестных значений {unk:.3f} > 1 % — дополнить таксономию до обучения")
    t2a = audit["targets"]["target_t2a_link_onset"]
    unobs = t2a["null_reasons"].get("unobserved_window", 0)
    if unobs > t2a["positive"] + t2a["negative"]:
        warn.append(f"- T2a: окон без событий ({unobs:,}) больше, чем размеченных — событийные каналы молчат; "
                    "coverage показывать отдельно")
    stage = manifest["stage"]
    moved = sum(v.get("rows_from_other_year_files", 0) for v in stage["years"].values())
    dups = sum(v.get("exact_duplicate_rows_dropped", 0) for v in stage["years"].values())
    rb = stage.get("row_balance", {})
    if rb:
        L_rb = (f"- баланс строк: источники {rb['source_rows']:,} = в панель {rb['staged_rows']:,} + точные дубли "
                f"{rb['exact_duplicates_dropped']:,} + строки заголовка {rb['header_rows_excluded']:,} + без года "
                f"{rb['unparseable_year_rows_excluded']:,} + вне режима {rb['skipped_outside_mode_years']:,}; "
                f"необъяснённых: **{rb['unexplained']:,}**")
        warn.append(L_rb)
    if manifest.get("missing_years"):
        warn.append("- **нет журналов за " + ", ".join(map(str, manifest["missing_years"])) + "** — панель неполная, "
                    "train фолдов ноутбука 15 без этого года")
    if moved:
        warn.append(f"- {moved:,} строк лежали в файле другого года — перенесены по дате события")
    if dups:
        warn.append(f"- удалено точных дублей строк: {dups:,}")
    L += warn or ["- нет"]
    sm = {}
    for fa in audit["per_file"].values():
        for stream, counts in fa.get("stream_mapping", {}).items():
            if stream == "events_in_no_stream_by_class":
                continue
            for k, v in counts.items():
                sm.setdefault(stream, {}).setdefault(k, 0)
                sm[stream][k] += v
    if sm:
        L += ["", f"## Потоки (stream mapping {manifest['panel_config'].get('stream_mapping_version', '?')}): секунды по состоянию", "",
              "| поток | F | O | A (конфликт) | A (неизвестно) | без сведений |", "|---|---|---|---|---|---|"]
        for stream, c in sm.items():
            L.append(f"| {stream} | {c.get('F', 0):,} | {c.get('O', 0):,} | {c.get('A_conflict', 0):,} | "
                     f"{c.get('A_unknown', 0):,} | {c.get('none', 0):,} |")
    if audit.get("sensitivity"):
        L += ["", "## Sensitivity-версии разметки (заранее объявлены; выбор не по validation-метрикам)", ""]
        for name, sv in audit["sensitivity"].items():
            L.append(f"**{name}** ({sv['config']}): изменений меток —")
            for t, ch in sv["label_changes"].items():
                if ch:
                    L.append(f"- `{t}`: " + ", ".join(f"{k} {v:,}" for k, v in ch.items()))
            L.append("")
    L += ["", "Метки — наблюдаемые события журнала (потеря связи, обесточивание, технические значения, газ ≥ 1 %), "
          "не подтверждённые физические отказы. Правила: docs/EVENT_PANEL_V3_RU.md.", ""]
    return "\n".join(L)


def run_event_panel_v3(cfg: dict[str, Any], log: Any = print) -> dict[str, Any]:
    import polars as pl

    t_start = time.time()
    out = Path(cfg["output_dir"])
    panel_dir = out / "event_panel_v3"
    shutil.rmtree(panel_dir, ignore_errors=True)
    panel_dir.mkdir(parents=True)
    src = discover_sources(Path(cfg["input_dir"]))
    if not src["journals"] or src["catalogue"] is None:
        return {"status": "stopped", "stop_reason": "не найдены ext-journal-YYYY.* или " + CATALOGUE_NAME
                + " во входах (Add Input → приватный dataset)"}
    smoke = cfg["mode"] == "SMOKE"
    years = cfg["smoke_years"] if smoke else None
    log(f"входы: годы {list(src['journals'])}, справочник найден; режим {cfg['mode']}")
    year_files, stage = stage_journals(pl, src["journals"], Path(cfg["stage_dir"]), years, log)
    if not year_files:
        return {"status": "stopped", "stop_reason": "после отбора годов не осталось журналов"}
    channel_filter = (pl.col("ид_канала_данных").hash(seed=7) % cfg["smoke_channel_share"] == 0) if smoke else None
    _, audit = build_event_panel(pl, year_files, src["catalogue"], Path(cfg["taxonomy_path"]), config=cfg["panel"],
                                 log=log, n_buckets=cfg["n_buckets"], out_dir=panel_dir, channel_filter=channel_filter)
    _truncate_unknown(audit)
    audit["sensitivity"] = {}
    key_cols = ["d_channel_key", "d_cutoff_date"] + TARGETS
    main_labels = pl.read_parquet([panel_dir / p for p in audit["part_files"]], columns=key_cols)
    for name in cfg.get("sensitivity_variants", []):
        vdir = Path(cfg["stage_dir"]) / f"sensitivity_{name}"
        shutil.rmtree(vdir, ignore_errors=True)
        vdir.mkdir(parents=True)
        log(f"sensitivity «{name}»: пересборка разметки")
        _, va = build_event_panel(pl, year_files, src["catalogue"], Path(cfg["taxonomy_path"]),
                                  config={**cfg["panel"], **SENSITIVITY_VARIANTS[name]}, log=lambda *a: None,
                                  n_buckets=cfg["n_buckets"], out_dir=vdir, channel_filter=channel_filter)
        var_labels = pl.read_parquet([vdir / p for p in va["part_files"]], columns=key_cols)
        audit["sensitivity"][name] = {"config": SENSITIVITY_VARIANTS[name],
                                      "label_changes": label_changes(pl, main_labels, var_labels),
                                      "targets": {t: {k: v for k, v in va["targets"][t].items() if k != "by_year"}
                                                  for t in TARGETS}}
        shutil.rmtree(vdir, ignore_errors=True)
    del main_labels
    parts = []
    for name in audit["part_files"]:
        p = panel_dir / name
        parts.append({"file": name, "bytes": p.stat().st_size, "sha256": file_sha256(p),
                      "rows": int(pl.scan_parquet(p).select(pl.len()).collect().item())})
    manifest = {
        "schema_version": SCHEMA_VERSION, "created_at_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
        "mode": cfg["mode"], "status": "completed_smoke_non_comparable" if smoke else "completed",
        "config": {k: v for k, v in cfg.items() if k not in ("input_dir", "output_dir", "stage_dir", "taxonomy_path")},
        "panel_config": audit["config"], "taxonomy_version": audit["taxonomy_version"],
        "taxonomy_sha256": audit["taxonomy_sha256"], "data_end": audit["data_end"],
        "catalogue": {"bytes": src["catalogue"].stat().st_size, "sha256": file_sha256(src["catalogue"]),
                      "candidates_found": src["catalogue_candidates"]},
        "sources_ignored_as_duplicates": src["ignored_duplicates"], "missing_years": src["missing_years"],
        "stage": stage, "parts": parts,
        "rows": audit["rows"], "channels": audit["channels"],
        "key_columns": ["d_channel_key", "d_cutoff_date"], "feature_columns": audit["feature_columns"],
        "target_columns": audit["target_columns"],
        "reason_columns": ["reason_" + t[len("target_"):] for t in audit["target_columns"]],
        "purge_column": "d_label_decision_end", "polars_version": pl.__version__,
        "runtime_s": round(time.time() - t_start, 1),
    }
    (panel_dir / "panel_manifest_v3.json").write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8")
    (out / "audit_event_panel_v3.json").write_text(json.dumps(audit, ensure_ascii=False, indent=2, default=str), encoding="utf-8")
    (out / "summary_event_panel_v3_ru.md").write_text(summary_markdown(manifest, audit), encoding="utf-8")
    if not cfg["keep_stage"]:
        shutil.rmtree(cfg["stage_dir"], ignore_errors=True)
    return {"status": manifest["status"], "rows": audit["rows"], "channels": audit["channels"],
            "runtime_s": manifest["runtime_s"], "panel_dir": str(panel_dir)}


## Самопроверки на синтетике


In [ ]:
import tempfile
SELF_TESTS = run_event_panel_self_tests(TAXONOMY_PATH, Path(tempfile.mkdtemp()))
for name, ok in SELF_TESTS.items():
    print("OK  " if ok else "FAIL", name)
assert all(SELF_TESTS.values()), "Самопроверки не пройдены"


## Проверка входов

Что найдено во входах ноутбука (имена и размеры файлов, без содержимого).


In [ ]:
CONFIG = make_config_v3(MODE, {"taxonomy_path": str(TAXONOMY_PATH), "n_buckets": N_BUCKETS,
                               "panel": {"tau_minutes": TAU_MINUTES},
                               **({} if RUN_SENSITIVITY else {"sensitivity_variants": []})})
SOURCES = discover_sources(Path(CONFIG["input_dir"]))
for line in describe_sources(SOURCES):
    print(line)
if not SOURCES["journals"] or SOURCES["catalogue"] is None:
    raise FileNotFoundError(
        "Ноутбуку 14 нужны сырые журналы ext-journal-YYYY (.7z/.csv/.parquet или папки Kaggle) и справочник каналов "
        "(колонки ид_канала_данных, тип_датчика) в Add Input. Обезличенная панель v2 не подходит — в ней нет истории "
        "событий. Проверенный корень входов: " + str(CONFIG["input_dir"]))


## Сборка панели


In [ ]:
RESULT = run_event_panel_v3(CONFIG)
print()
print("Статус:", RESULT["status"])
if RESULT.get("stop_reason"):
    raise RuntimeError("Панель не построена: " + RESULT["stop_reason"])


## Сводка для команды


In [ ]:
from IPython.display import Markdown, display
OUT = Path(CONFIG["output_dir"])
if RESULT["status"].startswith("completed"):
    display(Markdown((OUT / "summary_event_panel_v3_ru.md").read_text(encoding="utf-8")))


## Доля положительных по годам

2021 — переход системы мониторинга (FAQ): в ноутбуке 15 исключается из обучения и оценки.


In [ ]:
import pandas as pd
if RESULT["status"].startswith("completed"):
    audit = json.loads((OUT / "audit_event_panel_v3.json").read_text(encoding="utf-8"))
    rows = []
    for target, s in audit["targets"].items():
        for year, c in s["by_year"].items():
            lab = c["pos"] + c["neg"]
            rows.append({"цель": target.replace("target_", ""), "год": year, "1": c["pos"], "0": c["neg"],
                         "null": c["null"], "доля 1": round(c["pos"] / lab, 5) if lab else None})
    display(pd.DataFrame(rows).pivot_table(index="год", columns="цель", values="доля 1"))
    display(pd.DataFrame(rows).pivot_table(index="год", columns="цель", values="1", aggfunc="sum"))


In [ ]:
OUT = Path(CONFIG["output_dir"])
if RESULT["status"].startswith("completed"):
    files = sorted((OUT / "event_panel_v3").glob("*")) + [OUT / "audit_event_panel_v3.json", OUT / "summary_event_panel_v3_ru.md"]
    missing = [p.name for p in files if not p.is_file()]
    if missing:
        raise RuntimeError("Сборка завершилась без ожидаемых файлов: " + ", ".join(missing))
    print("Файлы для передачи команде и ноутбуку 15:")
    for p in files:
        print(f"- {p.relative_to(OUT)} ({p.stat().st_size / 2**20:.1f} МБ)")
    print("Сырые данные и промежуточные parquet удалены; в выходе только псевдонимизированная панель и агрегаты.")
else:
    print("Выходные файлы отсутствуют: панель не построена.")
